# 💰 Farm Financial — Model Training (Time-Series Random Forest)Dataset: `farm_financial_logs.csv`This notebook builds 7-day lag features and trains a **Random Forest Regressor** for time-series profit forecasting. It performs a chronological train/test split, evaluates the model (MAPE, MAE, R², explained variance), visualizes forecast vs. actual profit and residuals, and saves the trained model as `farm_financial_model.pkl`.

In [11]:
pip install matplotlib seaborn jupyter

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 69.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.5/77.5 kB 8.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.8/59.8 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 100.3 MB/s eta 0:00:00


# **Model Train Farm Financial**

In [ ]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_percentage_error, mean_absolute_error
import joblib

# 1. Load your financial ledger data
df_model3 = pd.read_csv('farm_financial_logs.csv')
df_model3['date'] = pd.to_datetime(df_model3['date'])

# 2. Build Time-Series Lag Features (Looking backward to predict forward)
for lag in range(1, 8):
    df_model3[f'profit_lag_{lag}'] = df_model3['daily_profit_pkr'].shift(lag)

# 3. Drop rows containing NaN values created by shifts
df_model3 = df_model3.dropna().reset_index(drop=True)

# 4. Separate features (X) from the target forecast (y)
# Features include historical lags and real-time operational scenario drivers
feature_cols = [col for col in df_model3.columns if 'lag_' in col] + ['sick_cow_count', 'total_milk_l', 'feed_cost_pkr']
X_finance = df_model3[feature_cols]
y_finance = df_model3['daily_profit_pkr']

print("✂️ Financial Time-Series Feature Engineering Complete!")
print(f"Total engineered training vectors: {X_finance.shape[0]} days")

In [ ]:
# Calculate split index for chronological order
split_idx = int(len(X_finance) * 0.8)

X_train_f, X_test_f = X_finance.iloc[:split_idx], X_finance.iloc[split_idx:]
y_train_f, y_test_f = y_finance.iloc[:split_idx], y_finance.iloc[split_idx:]

print(f"🗓️ Training on historical days 1-{split_idx}. Testing on future sequence days {split_idx+1}-{len(X_finance)}.")

# Initialize and train the Forecaster Engine
model_forecaster = RandomForestRegressor(n_estimators=150, random_state=42)
model_forecaster.fit(X_train_f, y_train_f)

print("✅ Forecaster Model training complete!")

In [ ]:
# Run future projections
y_pred_f = model_forecaster.predict(X_test_f)

# Compute standard error percentages
mape = mean_absolute_percentage_error(y_test_f, y_pred_f) * 100
mae_pkr = mean_absolute_error(y_test_f, y_pred_f)

# NEW: Compute overall goodness-of-fit model metrics
from sklearn.metrics import r2_score, explained_variance_score
r2_f = r2_score(y_test_f, y_pred_f)
exp_var = explained_variance_score(y_test_f, y_pred_f)

print("\n📉 --- Model 3 Evaluation Statistics ---")
print(f"Mean Absolute Percentage Error (MAPE): {mape:.2f}%")
print(f"Average Forecast Deviation (MAE):       {mae_pkr:.2f} PKR")
print(f"R² Score (Coefficient of Determination): {r2_f:.4f}")
print(f"Explained Variance Score:               {exp_var:.4f}")

# Update evaluation check based on your UI blueprint targets
if mape <= 8.4:
    print("\n🎉 Fantastic! Your forecaster accuracy hits or beats your exact target UI design goal (8.4% MAPE threshold)! ✅")
elif mape <= 10.0:
    print("\n👍 Good job! Your forecaster is highly functional and within an acceptable operational error margin. ✨")
else:
    print("\n⚠️ The error margin is slightly wide. Consider checking for missing data, adding more lag columns, or tuning hyper-parameters.")

In [ ]:
# Set clean, professional styling for financial validation dashboards
sns.set_theme(style="darkgrid")

# Create a clean 1-row, 2-column dashboard layout
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# --- CHART 1: Timeline Forecast vs. Actual Profit Tracks ---
# Reset index to treat consecutive test days as a clean chronological sequence
test_days = np.arange(len(y_test_f))

ax1.plot(test_days, y_test_f.values, label='Actual Historical Profit',
         color='#2ca02c', marker='o', lw=2)
ax1.plot(test_days, y_pred_f, label='AI Forecasted Profit',
         color='#ff7f0e', linestyle='--', marker='x', lw=2)

ax1.set_title('Time-Series Horizon: Actual vs. Forecasted Profit Track 💵', fontsize=13, pad=10)
ax1.set_xlabel('Consecutive Forecast Future Days', fontsize=11)
ax1.set_ylabel('Net Return (PKR)', fontsize=11)
ax1.legend(loc='upper right', frameon=True)

# --- CHART 2: Residual Error Distribution Analysis ---
# Calculate tracking errors (Residuals = Actual - Predicted)
residuals = y_test_f.values - y_pred_f

sns.histplot(residuals, kde=True, color='#d62728', bins=10, ax=ax2)
ax2.axvline(0, color='black', linestyle=':', lw=2, label='Zero Error Mark')

ax2.set_title('Residual Analysis: Prediction Deviation Spread 📉', fontsize=13, pad=10)
ax2.set_xlabel('Error Margin Deviation (PKR)', fontsize=11)
ax2.set_ylabel('Frequency Count (Days)')
ax2.legend(loc='upper left', frameon=True)

plt.tight_layout()
plt.show()

In [ ]:
import joblib
from google.colab import files

# Save the farm financial forecasting model
joblib.dump(model_forecaster, 'farm_financial_model.pkl')
print("💾 Farm Financial Forecasting Model serialized as 'farm_financial_model.pkl'!")

# Instantly download to your local computer
files.download('farm_financial_model.pkl')